# Gli esperimenti sul dataset estesoLa Fase 1 ha stabilito due cose sul dataset del paper. Che il residuo dell'autoencoderordina i cuscinetti **per esemplare** e non per stato: la dispersione fra i diciassetteesemplari vale un fattore due, la differenza fra sano e guasto il 9,5%. E che l'unicamodifica che sposta davvero la separazione e portare il segnale dentro il campo della SELUcon una costante unica, che alza l'area sotto la curva da 0,600 a 0,775.Qui si riprendono quelle due conclusioni e le si mette alla prova su dati piu grandi e conuna valutazione piu severa: ventinove cuscinetti invece di diciassette, quattro condizionioperative invece di una, e soprattutto la divisione fra addestramento e verifica fatta per**cuscinetti interi**. Nella replica gli esemplari di verifica erano gli stessidell'addestramento; qui il modello deve pronunciarsi su cuscinetti che non ha mai visto, chee la situazione reale in fabbrica.Il notebook e diviso in tre parti. La prima sceglie la configurazione dell'autoencoder. Laseconda misura i residui nei sei esperimenti previsti, e costa pochi minuti. La terzacompleta la catena fino alla rete convolutiva, ed e la parte cara.

In [ ]:
!pip -q install scipy scikit-learn

In [ ]:
import os, sys, gc, time, json, shutil, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

REPO = 'https://github.com/matpaol/MacchineEdAzionamentiExam'
possibili = ['.', '..', '../codice',
             '/content/drive/MyDrive/MacchineEdAzionamentiExam',
             '/content/MacchineEdAzionamentiExam']
percorso_codice = None
for c in possibili:
    if os.path.exists(os.path.join(c, 'funzioni.py')):
        percorso_codice = os.path.abspath(c)
        break
if percorso_codice is None:
    print('codice non trovato in locale, clono il repo')
    subprocess.run(['git', 'clone', '-q', REPO, '/content/MacchineEdAzionamentiExam'], check=True)
    percorso_codice = '/content/MacchineEdAzionamentiExam'
sys.path.insert(0, percorso_codice)
import config
import funzioni as f

f.stile_grafici()
P = config.percorsi(sottocartella='05_esperimenti_esteso')
dev = f.dispositivo()
seme = 0

# quali esperimenti arrivano fino alla rete convolutiva. Ognuno costa mezz'ora
# scarsa: si parte dai due che rispondono alle domande principali, gli altri si
# aggiungono qui se il tempo lo consente.
CNN_DA_ESEGUIRE = ['soli_reali', 'artificiale_verso_reale']

print('codice da', percorso_codice)
print('dispositivo', dev)

## Il datasetI frame stanno su Drive, che su Colab e un disco di rete: leggerne 3,5 GB a pezzi sparsisarebbe lentissimo. Si copiano quindi una volta sul disco locale della macchina e poi siaprono in modalita mappata, cosi la memoria non se ne accorge e le letture sono veloci.

In [ ]:
sorgente = os.path.join(P['dataset'], 'frame.npy')
locale = '/content/frame.npy'
if not os.path.exists(locale):
    print('copio il dataset sul disco locale...')
    partenza = time.time()
    shutil.copy(sorgente, locale)
    print('copiato in', round(time.time() - partenza), 's')

X = np.load(locale, mmap_mode='r')
anagrafica = pd.read_csv(os.path.join(P['dataset'], 'anagrafica.csv'))
parametri = json.load(open(os.path.join(P['dataset'], 'parametri.json')))

print('frame:', X.shape, '| standardizzati per frame:',
      parametri['standardizzazione_per_frame'])
print('cuscinetti:', anagrafica['cuscinetto'].nunique(),
      '| registrazioni:', anagrafica['registrazione'].nunique())
print()
print(anagrafica.groupby('regime').size().to_string())
print()
print('righe dell anagrafica e frame della matrice coincidono:',
      len(anagrafica) == len(X))

## La scala del segnaleIl fattore si calcola come nella Fase 1: dal picco piu alto dei soli frame sani diaddestramento, portato a 1,5 per lasciare un margine sotto il pavimento della SELU.C'e pero una differenza che va misurata prima di procedere. Nella replica c'era una solacondizione operativa; qui ce ne sono quattro, e la corrente cambia molto da una all'altra.Il troncamento della SELU dipende dall'ampiezza, quindi non agisce allo stesso mododappertutto, e la cella lo verifica regime per regime.

In [ ]:
sani_dae = anagrafica['cuscinetto'].isin(config.SANI_DAE_TRAIN).values
frame_sani = np.asarray(X[np.flatnonzero(sani_dae)])
fattore, massimo = f.scala_globale(frame_sani)

print('massimo assoluto dei sani di addestramento:', round(massimo, 4), 'A')
print('fattore di scala:', round(fattore, 6))
print()

righe = []
for regime, gruppo in anagrafica.groupby('regime'):
    campione = np.asarray(X[np.flatnonzero((anagrafica['regime'] == regime).values)[:4000]])
    righe.append({'regime': regime,
                  'frame': int((anagrafica['regime'] == regime).sum()),
                  'rms': float(np.sqrt(np.mean(campione ** 2))),
                  'picco': float(np.max(np.abs(campione))),
                  'sotto_pavimento_pct': 100 * float(np.mean(campione < config.PAVIMENTO_SELU)),
                  'sotto_dopo_scala_pct': 100 * float(np.mean(campione * fattore
                                                              < config.PAVIMENTO_SELU))})
tabella_regimi = pd.DataFrame(righe)
print(tabella_regimi.round(3).to_string(index=False))
print()
print('la scala azzera il troncamento dappertutto:',
      bool(tabella_regimi['sotto_dopo_scala_pct'].max() < 0.001))
del frame_sani
gc.collect()

## Come si misura un esperimentoOgni esperimento addestra un autoencoder sui soli cuscinetti sani che il protocollo gliassegna, poi calcola il residuo su un insieme di verifica e lo misura sempre allo stessomodo: residuo medio per classe, rapporto sul sano, area sotto la curva, e il dettaglio persingolo esemplare, che nella Fase 1 si e rivelato il numero piu informativo.L'autoencoder usa la catena del paper adattata all'ingresso da 4273 campioni, con arrestoanticipato a pazienza venti, che la Fase 1 ha mostrato equivalente alle cinquecento epochefisse.

In [ ]:
DIMENSIONI = [config.LUNGHEZZA_GIRO, 1280, 640, 320, 128, 32,
              128, 320, 640, 1280, config.LUNGHEZZA_GIRO]
PAZIENZA = 20


def indici(cuscinetti, regimi=None):
    """Posizioni dei frame che appartengono a quei cuscinetti e a quei regimi."""
    m = anagrafica['cuscinetto'].isin(cuscinetti).values
    if regimi is not None:
        m = m & anagrafica['regime'].isin(regimi).values
    return np.flatnonzero(m)


def addestra(cuscinetti_train, cuscinetti_val, regimi=None, scala=True,
             epoche=500, pazienza=PAZIENZA, etichetta=''):
    """Autoencoder sui soli sani indicati. Restituisce (modello, fattore, curve)."""
    k = 1.0
    tr = np.asarray(X[indici(cuscinetti_train, regimi)], dtype=np.float32)
    if scala:
        k, _ = f.scala_globale(tr)
        tr = tr * k
    va = np.asarray(X[indici(cuscinetti_val, regimi)], dtype=np.float32) * k

    print(etichetta, '| addestramento', tr.shape[0], 'frame | validazione', va.shape[0])
    modello, c_tr, c_va = f.addestra_dae(tr, va, uscita_selu=True, dimensioni=DIMENSIONI,
                                         epoche=epoche, lotto=256, passo=3e-4,
                                         pazienza=pazienza, seme=seme, dev=dev,
                                         stampa_ogni=25)
    del tr, va
    gc.collect()
    return modello, k, (c_tr, c_va)


def misura(modello, k, cuscinetti, regimi=None, etichetta=''):
    """Residuo per classe, rapporti, AUC e dettaglio per esemplare."""
    posizioni = indici(cuscinetti, regimi)
    parte = anagrafica.iloc[posizioni]
    mse = np.empty(len(posizioni), dtype=np.float64)
    for i in range(0, len(posizioni), 2048):
        blocco = np.asarray(X[posizioni[i:i + 2048]], dtype=np.float32) * k
        residuo = f.calcola_residui(modello, blocco, dev=dev) / k
        mse[i:i + len(blocco)] = f.mse_per_frame(residuo)

    classi = parte['classe'].values
    per_classe = [float(np.mean(mse[classi == c])) if (classi == c).any() else np.nan
                  for c in (0, 1, 2)]
    auc = f.auc_residuo(mse, classi)
    dettaglio = f.residuo_per_cuscinetto(mse, parte['cuscinetto'].values, classi)

    print('  ', etichetta, '|', len(posizioni), 'frame di verifica su',
          parte['cuscinetto'].nunique(), 'cuscinetti')
    riga = {'esperimento': etichetta,
            'residuo_sano': per_classe[0],
            'rapporto_esterno': per_classe[1] / per_classe[0],
            'rapporto_interno': per_classe[2] / per_classe[0],
            'auc_esterno': auc['esterno'], 'auc_interno': auc['interno'],
            'auc_media': auc['media'],
            'dispersione_esemplari': float(dettaglio['media'].max() / dettaglio['media'].min())}
    print('   rapporti {:.3f} / {:.3f}   AUC {:.4f}   dispersione fra esemplari {:.2f}'.format(
        riga['rapporto_esterno'], riga['rapporto_interno'],
        riga['auc_media'], riga['dispersione_esemplari']))
    return riga, dettaglio, mse


risultati = []
dettagli = {}

## Parte prima. La scala serve anche qui?La Fase 1 ha trovato nella scala globale l'unica modifica che sposta la separazione. Ma laFase 1 aveva un regime solo, dove il troncamento colpiva un quarto dei campioni. Qui iregimi sono quattro e il troncamento non e uniforme, quindi la domanda va rifatta.Si addestrano due autoencoder identici in tutto tranne la scala del segnale, sugli stessitre cuscinetti sani e su tutti e quattro i regimi, e si misurano sui cuscinetti a dannoreale, che nessuno dei due ha mai visto.

In [ ]:
verifica_configurazioni = config.REALI[1] + config.REALI[2] + config.SANI_TEST

for etichetta, scala in [('segnale non scalato', False), ('scala globale', True)]:
    modello, k, curve = addestra(config.SANI_DAE_TRAIN, config.SANI_DAE_VAL,
                                 scala=scala, etichetta=etichetta)
    riga, dettaglio, _ = misura(modello, k, verifica_configurazioni, etichetta=etichetta)
    risultati.append(riga)
    dettagli[etichetta] = dettaglio
    del modello
    gc.collect()
    print()

## Parte seconda. I sei esperimenti, misurati sui residuiOgni esperimento cambia una cosa sola rispetto agli altri, e ciascuno risponde a una domandache la replica non poteva porsi.**Soli danni reali** usa gli stessi cuscinetti del paper ma con gli esemplari separati fraaddestramento e verifica. Non va chiamato replica: i regimi sono quattro e la divisione epiu severa.**Artificiale verso reale** e l'esperimento principale del dataset esteso: si impara sudanni fatti a macchina, che costano poco, e si verifica su danni cresciuti in prove di vitaaccelerate, che costano molto. E la domanda della manutenzione predittiva.**Velocita** e **coppia** chiedono se il residuo rappresenti il guasto oppure il punto difunzionamento. Entrambi addestrano su una condizione sola e verificano su quella e suun'altra. Sono costruiti in modo da cambiare **una variabile per volta**: per la velocita siconfrontano 1500 e 900 giri a coppia piena, per la coppia si confrontano coppia piena eridotta a 1500 giri. Il piano originale prevedeva di addestrare sui tre regimi a 1500 giri,ma quelli differiscono anche per coppia e i due effetti resterebbero confusi.**Rodaggio** confronta due definizioni di normale: cuscinetti sani con storie molto diversecontro cuscinetti sani poco usati. Qui l'arresto anticipato viene disattivato e le epochefissate a quelle trovate prima, perche le due varianti hanno insiemi diversi e un arrestogovernato da insiemi diversi non le renderebbe confrontabili.**Severita** addestra sui danni artificiali estesi e verifica su quelli incipienti.

In [ ]:
REGIMI_PIENI = ['N09_M07_F10', 'N15_M07_F04', 'N15_M07_F10']
UN_REGIME = ['N15_M07_F10']

esperimenti = [
    ('soli reali',            config.SANI_DAE_TRAIN, config.SANI_DAE_VAL, None,
     config.INSIEMI['soli_reali']['test'], None),
    ('artificiale -> reale',  config.SANI_DAE_TRAIN, config.SANI_DAE_VAL, None,
     config.INSIEMI['artificiale_verso_reale']['test'], None),
    ('severita',              config.SANI_DAE_TRAIN, config.SANI_DAE_VAL, None,
     config.INSIEMI['severita']['test'], None),
    ('velocita, 1500 giri',   config.SANI_DAE_TRAIN, config.SANI_DAE_VAL, UN_REGIME,
     config.REALI[1] + config.REALI[2] + config.SANI_TEST, UN_REGIME),
    ('velocita, 900 giri',    None, None, None,
     config.REALI[1] + config.REALI[2] + config.SANI_TEST, ['N09_M07_F10']),
    ('coppia piena',          None, None, None,
     config.REALI[1] + config.REALI[2] + config.SANI_TEST, UN_REGIME),
    ('coppia ridotta',        None, None, None,
     config.REALI[1] + config.REALI[2] + config.SANI_TEST, ['N15_M01_F10']),
]

modello_corrente, k_corrente = None, 1.0
for etichetta, tr, va, reg_tr, test, reg_test in esperimenti:
    if tr is not None:                      # None: riusa l'autoencoder precedente
        if modello_corrente is not None:
            del modello_corrente
            gc.collect()
        modello_corrente, k_corrente, _ = addestra(tr, va, regimi=reg_tr, etichetta=etichetta)
    riga, dettaglio, _ = misura(modello_corrente, k_corrente, test, regimi=reg_test,
                                etichetta=etichetta)
    risultati.append(riga)
    dettagli[etichetta] = dettaglio
    print()

In [ ]:
# Il rodaggio: due definizioni di normale, a epoche fisse per essere confrontabili
epoche_fisse = 60

for etichetta, sani in [('rodaggio eterogeneo', config.SANI_ETEROGENEI),
                        ('rodaggio poco usato', config.SANI_POCO_RODATI)]:
    modello, k, _ = addestra(sani, sani, epoche=epoche_fisse, pazienza=None,
                             etichetta=etichetta)
    non_visti = [b for b in config.CUSCINETTI_PER_CLASSE[0] if b not in sani]
    riga, dettaglio, _ = misura(modello, k,
                                config.REALI[1] + config.REALI[2] + non_visti,
                                etichetta=etichetta)
    riga['sani_mai_visti'] = ' '.join(non_visti)
    risultati.append(riga)
    dettagli[etichetta] = dettaglio
    del modello
    gc.collect()
    print()

In [ ]:
confronto = pd.DataFrame(risultati)
colonne = ['esperimento', 'residuo_sano', 'rapporto_esterno', 'rapporto_interno',
           'auc_esterno', 'auc_interno', 'auc_media', 'dispersione_esemplari']
print(confronto[colonne].round(4).to_string(index=False))
print()
print('per confronto, la Fase 1 sul dataset del paper con la scala globale:')
print('   rapporti 1,414 / 1,356   AUC media 0,775')
print('e i valori dichiarati dal paper: rapporti 3,712 / 4,606')

## Parte terza. La rete convolutivaQui la catena si chiude. Il residuo dei blocchi da quattordici giri diventa l'ingresso dellarete, che viene addestrata sui cuscinetti previsti dal protocollo e verificata su esemplarimai visti.Il conteggio viene fatto su tre unita diverse a partire dalle stesse previsioni. Sul**blocco**, che e quello che riporta il paper ed e il piu ottimistico, perche blocchi dellastessa registrazione sono quasi copie l'uno dell'altro. Sulla **registrazione**, per voto dimaggioranza fra i suoi blocchi. E sul **cuscinetto**, che e l'unico che risponde alla domandaindustriale: quanti esemplari sono stati diagnosticati correttamente.

In [ ]:
def blocchi_di(cuscinetti, regimi=None):
    """Indici dei blocchi e loro anagrafica, per un insieme di cuscinetti."""
    m = anagrafica['cuscinetto'].isin(cuscinetti).values
    if regimi is not None:
        m = m & anagrafica['regime'].isin(regimi).values
    return f.costruisci_blocchi(anagrafica[m], config.GIRI_PER_BLOCCO)


def residui_blocchi(modello, k, indici_blocchi):
    """Residuo di ogni blocco, riportato in ampere."""
    n, giri = indici_blocchi.shape
    uscita = np.empty((n, giri * X.shape[1]), dtype=np.float32)
    for i in range(0, n, 16):
        gruppo = indici_blocchi[i:i + 16]
        pezzo = np.asarray(X[gruppo.ravel()], dtype=np.float32) * k
        residuo = f.calcola_residui(modello, pezzo, dev=dev) / k
        uscita[i:i + len(gruppo)] = residuo.reshape(len(gruppo), -1)
    return uscita


esiti_cnn = {}
for nome in CNN_DA_ESEGUIRE:
    ins = config.INSIEMI[nome]
    print()
    print('===', nome)
    modello_dae, k, _ = addestra(ins['dae_train'], ins['dae_val'], etichetta='DAE di ' + nome)

    insiemi = {}
    for parte in ['train', 'val', 'test']:
        if len(ins[parte]) == 0:
            continue
        idx, ana = blocchi_di(ins[parte])
        insiemi[parte] = (residui_blocchi(modello_dae, k, idx),
                          ana['classe'].values.astype(np.int64), ana)
        print('   ', parte, len(idx), 'blocchi da', idx.shape[1], 'giri')
    del modello_dae
    gc.collect()

    if 'val' not in insiemi:                 # severita: nessuna validazione per costruzione
        insiemi['val'] = insiemi['train']
        pazienza_cnn = None
    else:
        pazienza_cnn = PAZIENZA

    modello_cnn, storia = f.addestra_cnn_insiemi(
        insiemi['train'][0], insiemi['train'][1],
        insiemi['val'][0], insiemi['val'][1],
        epoche=500, lotto=64, passo=3e-4, pazienza=pazienza_cnn,
        seme=seme, dev=dev, stampa_ogni=10, etichetta=nome)

    previsioni = f.prevedi_cnn(modello_cnn, insiemi['test'][0], dev=dev)
    conteggio = f.conteggio_tre_livelli(previsioni, insiemi['test'][2])
    metriche = f.metriche(insiemi['test'][1], previsioni)

    print()
    for livello in ['blocchi', 'registrazioni', 'cuscinetti']:
        c = conteggio[livello]
        print(f"   {livello:14s} {c['giusti']:5d}/{c['totale']:<5d} = {100 * c['accuratezza']:.2f} %")
    print()
    print(metriche['report'])
    print(pd.DataFrame(metriche['confusione'], index=config.NOMI_CLASSI,
                       columns=config.NOMI_CLASSI).to_string())
    print()
    dettaglio = conteggio['dettaglio_cuscinetti'].copy()
    dettaglio['classe_vera'] = [config.NOMI_CLASSI[c] for c in dettaglio['vera']]
    dettaglio['classe_prevista'] = [config.NOMI_CLASSI[c] for c in dettaglio['prevista']]
    print(dettaglio[['cuscinetto', 'classe_vera', 'classe_prevista', 'blocchi',
                     'quota_voto']].to_string(index=False))

    esiti_cnn[nome] = {'conteggio': conteggio, 'metriche': metriche,
                       'storia': storia, 'dettaglio': dettaglio}
    insiemi.clear()
    del insiemi, modello_cnn
    gc.collect()

## Riepilogo

In [ ]:
righe = []
for nome, e in esiti_cnn.items():
    riga = {'esperimento': nome}
    for livello in ['blocchi', 'registrazioni', 'cuscinetti']:
        riga[livello] = 100 * e['conteggio'][livello]['accuratezza']
        riga[livello + '_totale'] = e['conteggio'][livello]['totale']
    riga['macro_f1'] = e['metriche']['f1']
    riga['epoche'] = e['storia']['epoche']
    riga['minuti'] = e['storia']['durata_s'] / 60
    righe.append(riga)

tabella_cnn = pd.DataFrame(righe)
if len(tabella_cnn):
    print(tabella_cnn.round(2).to_string(index=False))
    print()
    print('per confronto, la replica del Capitolo 6 con la suddivisione permeabile del paper:')
    print('   46,86 % sui segmenti, e il paper ne dichiara 99,60')

In [ ]:
fig, assi = plt.subplots(1, 2, figsize=(12, 4.4))

ordine = confronto.sort_values('auc_media')
posizioni = np.arange(len(ordine))
assi[0].barh(posizioni, ordine['auc_media'], color=f.COLORI['nostro'])
assi[0].axvline(0.5, color=f.COLORI['neutro'], ls='--', lw=1)
assi[0].axvline(0.775, color=f.COLORI['accento'], ls='--', lw=1,
                label='Fase 1, dataset del paper')
assi[0].set_yticks(posizioni); assi[0].set_yticklabels(ordine['esperimento'], fontsize=8)
assi[0].set_xlim(0.4, 1.0); assi[0].set_xlabel('AUC media')
assi[0].set_title('Separazione dei residui, per esperimento', fontsize=10)
assi[0].legend(fontsize=7)

assi[1].barh(posizioni, ordine['dispersione_esemplari'], color=f.COLORI['interno'])
assi[1].axvline(1.0, color=f.COLORI['neutro'], ls='--', lw=1)
assi[1].set_yticks(posizioni); assi[1].set_yticklabels([])
assi[1].set_xlabel('massimo / minimo fra esemplari')
assi[1].set_title('Quanto il residuo dipende dal singolo cuscinetto', fontsize=10)

f.salva_figura(fig, 'confronto_esperimenti', P['figure'])
plt.show()

In [ ]:
f.salva_tabella(confronto, 'confronto_esperimenti', P['tabelle'])
f.salva_tabella(tabella_regimi, 'troncamento_per_regime', P['tabelle'])
if len(tabella_cnn):
    f.salva_tabella(tabella_cnn, 'risultati_cnn', P['tabelle'])
def nome_file(testo):
    """Nome di file leggibile: solo lettere, cifre e trattini bassi."""
    pulito = ''.join(c if c.isalnum() else '_' for c in testo)
    while '__' in pulito:
        pulito = pulito.replace('__', '_')
    return pulito.strip('_')


for nome, d in dettagli.items():
    f.salva_tabella(d, 'residui_' + nome_file(nome), P['tabelle'])
for nome, e in esiti_cnn.items():
    f.salva_tabella(e['dettaglio'], 'cuscinetti_' + nome_file(nome), P['tabelle'])

for cartella in [P['figure'], P['tabelle']]:
    for nome in sorted(os.listdir(cartella)):
        percorso = os.path.join(cartella, nome)
        if os.path.isfile(percorso):
            print(nome, round(os.path.getsize(percorso) / 1e6, 3), 'MB')